# 00 — Regression Check vs Camera-Ready

Loads a camera-ready pickle from `ICML_2025_Workshop_Submission_Artifacts/pickles/`, extracts its actor state, and replays it through the cleaned pipeline. The cleaned `Actor` class should accept the legacy state dict, and applying the actor to a fresh latent batch should produce a mean reward consistent with the camera-ready run.

This is the **semantic-equivalence check**: it verifies the refactor did not change network architecture, hyperparameters, or evaluation protocol in any way that breaks reproducibility.

If the cleaned pipeline ever evolves (e.g., multiprocessing decode, different reward weight defaults), this notebook flags it.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

from lisardd.agents.networks import Actor, ActorReinforce
from lisardd.decoding.safe_decode import safe_decode_batch
from lisardd.generators.hiervae_wrapper import HierVAEGenerator
from lisardd.io import load_legacy_pickle
from lisardd.rewards import reward_binding_affinity
from lisardd.scoring.mgraphdta_wrapper import MGraphDTAScorer
from lisardd.targets import get_sequence

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Configuration

Point at one camera-ready pickle. The default below targets the JNK3 binding-affinity-only PPO run; change `pickle_name` to test other configurations from the 12-run matrix.

In [ ]:
pickle_name = "ppo_binding_jkn3_normalize.pkl"  # note typo preserved from camera-ready filename
target_name = "jnk3"
latent_dim = 32
hidden_dim = 256
n_samples = 100
seed = 42

pickle_path = repo_root / "ICML_2025_Workshop_Submission_Artifacts" / "pickles" / pickle_name
print("pickle exists:", pickle_path.exists())
print("pickle path:", pickle_path)

In [ ]:
data = load_legacy_pickle(pickle_path)
print("keys:", list(data.keys()))
print("hyperparameters:", data.get("hyperparameters"))

## Load actor from pickle into the cleaned `Actor` class

If the architecture matches, `load_state_dict` should succeed without missing/unexpected keys.

In [ ]:
actor = Actor(latent_dim, hidden_dim).to(device)
incompat = actor.load_state_dict(data["actor_state_dict"], strict=True)
print("missing keys:", incompat.missing_keys)
print("unexpected keys:", incompat.unexpected_keys)
actor.eval()

## Apply actor to a fresh latent batch and score with MGraphDTA

With a fixed seed, this should produce a mean reward in a similar range to the camera-ready averages stored in `data['avg_obj_scores_ppo']`.

In [ ]:
torch.manual_seed(seed)

generator = HierVAEGenerator(
    vocab_path=repo_root / "data" / "chembl" / "recovered_vocab_2000.txt",
    ckpt_path=repo_root / "vae_model" / "vae_model.ckpt",
    device=device,
)

scorer = MGraphDTAScorer(
    target_protein=get_sequence(target_name),
    ckpt_path=repo_root / "score_model_weights" / "best_scoring_model.pt",
    device=device,
)
reward_fn = reward_binding_affinity(scorer)

In [ ]:
with torch.no_grad():
    z_base = torch.randn(n_samples, 3 * latent_dim, device=device)
    mu, std = actor(z_base)
    z_perturbed = torch.distributions.MultivariateNormal(mu, torch.diag_embed(std)).sample()
    z_chunks = torch.chunk(z_perturbed, 3, dim=1)
    smiles, valid = safe_decode_batch(generator.decoder, z_chunks, greedy=True)

valid_mask = torch.tensor(valid, device=device, dtype=torch.bool)
valid_smiles = [s for s, v in zip(smiles, valid) if v]

rewards = torch.zeros(n_samples, device=device)
rewards[~valid_mask] = -1
rewards[valid_mask] = reward_fn(valid_smiles)

print(f"Valid SMILES: {len(valid_smiles)}/{n_samples}")
print(f"Mean reward (replay): {rewards.mean().item():.4f}")

In [ ]:
import numpy as np

last10 = data["avg_obj_scores_ppo"][-10:]
print(f"Camera-ready avg over last 10 epochs: {np.mean(last10):.4f}")
print(f"Replay mean reward: {rewards.mean().item():.4f}")
print("NOTE: replay uses raw pKd; camera-ready avg is post-normalization, so values are NOT directly comparable.")
print("What to check: replay reward should be in a plausible pKd range (e.g., 5-12) for a trained policy.")

## Visualize the policy's preferred molecules

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

sorted_idx = torch.argsort(rewards, descending=True).tolist()
top_smiles = [smiles[i] for i in sorted_idx[:9] if smiles[i] is not None]
mols = [Chem.MolFromSmiles(s) for s in top_smiles]
legends = [f"pKd={rewards[i].item():.2f}" for i in sorted_idx[:9] if smiles[i] is not None]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=legends)